In [ ]:
%matplotlib widget

import ipywidgets as widgets
from ipywidgets import HBox, VBox, Box, Layout, Output, Label
from IPython.display import display, Latex, Markdown, Image, Math

import params as st
from model import nonlinSys, stationarySys
from feedforward import trajectory, feedForward, feedBack

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
def make_box_layout():
     return widgets.Layout(
        border='solid 1px black',
        margin='0px 5px 5px 0px',
        padding='2px 2px 2px 2px'
     )

# Eingangs-Ausgangslinearisierung des Zweitanksystems

In [ ]:
imag = Image("../images/twoTank.png", width=250)

outL = Output()
outR = Output()

with outL:
    display(imag)
with outR:
    display(Markdown(r"""
- Zwei Tanks in Reihe, Eingang Zufluss $u(t) \geq 0$ der Pumpe in Tank 1, Ausgang Füllstand $y(t) = h_1(t)$ von Tank 1
- Ausfluss nach Torricelli $q_i(h_i) = a_i \sqrt{2 g h_i}$
  \begin{align*}
      A_1 \dot{h}_1(t) & = u(t) - a_1 \sqrt{2 g h_1(t)}, &
      A_2 \dot{h}_2(t) & = a_1 \sqrt{2 g h_1(t)} - a_2 \sqrt{2 g h_2(t)}
  \end{align*}
- Systemdimension $n = 2$, relativer Grad $r = 1$, also verbleibt eine eindimensionale interne Dynamik
    """))
cols = HBox([outL, outR], layout=Layout(display='flex', flex_flow='row', justify_content='space-around', align_items='center'))
display(cols)

## Relativer Grad und Byrnes-Isidori-Normalform

Eingangsaffine Darstellung $\dot{\boldsymbol{x}} = \boldsymbol{f}(\boldsymbol{x}) + \boldsymbol{g}(\boldsymbol{x}) u$, $y = h(\boldsymbol{x})$ mit $\boldsymbol{x} = (h_1, h_2)^\intercal$

In [ ]:
h1, h2, u = sp.symbols('h_1 h_2 u', positive=True)
A1, A2, a1, a2, g = sp.symbols('A_1 A_2 a_1 a_2 g', positive=True)

x = sp.Matrix([h1, h2])
f = sp.Matrix([-a1 * sp.sqrt(2 * g * h1) / A1,
               (a1 * sp.sqrt(2 * g * h1) - a2 * sp.sqrt(2 * g * h2)) / A2])
gv = sp.Matrix([1 / A1, 0])

def lie(fun, vec):
    return sp.simplify((sp.Matrix([fun]).jacobian(x) * vec)[0])

display(Math(r"\boldsymbol{f}(\boldsymbol{x}) = " + sp.latex(f) + r",\quad \boldsymbol{g}(\boldsymbol{x}) = " + sp.latex(gv)))

In [ ]:
y = h1
display(Math(r"L_{\boldsymbol{g}} h(\boldsymbol{x}) = " + sp.latex(lie(y, gv)) + r" \neq 0 \quad\Rightarrow\quad r = 1"))

Als Koordinaten der Normalform eignen sich $\xi = h(\boldsymbol{x}) = h_1$ und $\eta = h_2$, da $L_{\boldsymbol{g}} \eta = 0$ gilt.
Damit lautet die Byrnes-Isidori-Normalform
\begin{align*}
    \dot{\xi} & = L_{\boldsymbol{f}} h + L_{\boldsymbol{g}} h\, u = \frac{1}{A_1}\left(u - a_1 \sqrt{2 g \xi}\right), &
    \dot{\eta} & = \frac{1}{A_2}\left(a_1 \sqrt{2 g \xi} - a_2 \sqrt{2 g \eta}\right).
\end{align*}
Die Nulldynamik ergibt sich für $y(t) \equiv y^*$, also $\xi = y^*$:

In [ ]:
yS, eta = sp.symbols('y^* eta', positive=True)
zeroDyn = f[1].subs({h1: yS, h2: eta})
etaS = sp.solve(zeroDyn, eta)[0]
lam = sp.simplify(sp.diff(zeroDyn, eta).subs(eta, etaS))

display(Math(r"\dot{\eta} = " + sp.latex(zeroDyn)))
display(Math(r"\eta^* = " + sp.latex(etaS) + r",\qquad \left.\frac{\partial \dot{\eta}}{\partial \eta}\right|_{\eta^*} = " + sp.latex(lam) + r" < 0"))

Die Ruhelage der Nulldynamik ist asymptotisch stabil, das System ist mit diesem Ausgang **minimalphasig**.
Der Füllstand $h_2$ wird von Steuerung und Regelung nicht vorgegeben, er stellt sich über die interne Dynamik ein und bleibt beschränkt.

**Vergleich:** Mit dem Ausgang $y = h_2$ erhält man stattdessen

In [ ]:
y2 = h2
Lfy2 = lie(y2, f)
display(Math(r"L_{\boldsymbol{g}} h_2 = " + sp.latex(lie(y2, gv)) + r",\qquad L_{\boldsymbol{g}} L_{\boldsymbol{f}} h_2 = " + sp.latex(lie(Lfy2, gv)) + r" \neq 0 \quad\Rightarrow\quad r = 2 = n"))

In diesem Fall gibt es keine interne Dynamik und $h_2$ ist ein flacher Ausgang.

## Steuerung und Regelung

Steuerung | | Regelung
----------------------|----|------------------------
\begin{align*}
    u_\mathrm{d}(t) & = A_1 \dot{y}_\mathrm{r}(t) + a_1 \sqrt{2 g\, y_\mathrm{r}(t)}
\end{align*} | | \begin{align*}
    u(t) & = a_1 \sqrt{2 g\, h_1(t)} + A_1 v(t), \\
    v(t) & = \dot{y}_\mathrm{r}(t) - k \left(y(t) - y_\mathrm{r}(t)\right), \quad k > 0
\end{align*}

Für die Steuerung muss die Referenz $y_\mathrm{r} \in C^1$ sein. Im geschlossenen Kreis gilt für den Folgefehler $e = y - y_\mathrm{r}$ die lineare Fehlerdynamik $\dot{e} = -k\, e$.
Um einen Modellfehler zu untersuchen, kann der Ablaufquerschnitt $a_1$ in der Simulation gegenüber dem Entwurfsmodell um $\Delta a_1$ verändert werden.

**Definition Parameter**

In [ ]:
tSim = np.linspace(0, 400, 801)
t0 = 20
y0 = 0.2

## Simulation

In [ ]:
def control(t, x, mode, k, ffArgs):
    if mode == 'Steuerung':
        return feedForward(t, *ffArgs)[1]
    else:
        return feedBack(t, x[0], k, *ffArgs)

def closedLoop(t, x, mode, k, ffArgs, params):
    u = lambda t: control(t, x, mode, k, ffArgs)
    return nonlinSys(t, x, u, params)

In [ ]:
output = widgets.Output()

with output:
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.subplots_adjust(bottom=0.07, top=0.93, left=0.1, right=0.95, hspace=0.1)

ax1.set_xlim([0, tSim[-1]])
for ax in [ax1, ax2, ax3]:
    ax.grid()
ax1.set_ylabel(r"$y = h_1$ in m")
ax2.set_ylabel(r"$h_2$ in m")
ax3.set_ylabel(r"$u$ in l/s")
ax3.set_xlabel(r"$t$ in s")

h1Ref, = ax1.plot([], [], 'C0--', label=r'Referenz $y_\mathrm{r}$')
h1Sys, = ax1.plot([], [], 'C2', label=r'System')
h2Stat, = ax2.plot([], [], 'C1--', label=r'Ruhelage $\eta^*(y_\mathrm{r})$')
h2Sys, = ax2.plot([], [], 'C2')
uFF, = ax3.plot([], [], 'C0--', label=r'Steuerung $u_\mathrm{d}$')
uSys, = ax3.plot([], [], 'C3', label=r'Stellgröße $u$')

handlesAx, labelsAx = [], []
for ax in [ax1, ax2, ax3]:
    h, la = ax.get_legend_handles_labels()
    handlesAx += h
    labelsAx += la

fig.legend(handlesAx, labelsAx,
           bbox_to_anchor=(0.1, 0.94, 0.85, .15), loc=3,
           ncol=5, mode="expand", borderaxespad=0., framealpha=0.5)

sliderYd = widgets.FloatSlider(value=0.1,
                               min=-0.15,
                               max=0.15,
                               step=0.01,
                               description=r"$y_\text{d}$")
sliderT = widgets.FloatSlider(value=100,
                              min=20,
                              max=200,
                              step=10,
                              description=r"$T$")
sliderK = widgets.FloatSlider(value=0.05,
                              min=0.01,
                              max=0.5,
                              step=0.01,
                              description=r"$k$")
sliderDh = widgets.FloatSlider(value=0,
                               min=-0.1,
                               max=0.1,
                               step=0.01,
                               description=r"$\Delta h_1(0)$")
sliderDa = widgets.FloatSlider(value=0,
                               min=-20,
                               max=20,
                               step=1,
                               description=r"$\Delta a_1$ in %")

radioU = widgets.RadioButtons(options=['Steuerung', 'Steuerung und Regelung'],
                              value='Steuerung und Regelung',
                              disabled=False)

def calcSystem(_):
    mode = radioU.value
    k = sliderK.value
    ffArgs = (y0, sliderYd.value, t0, sliderT.value)
    params = st.g, st.A1, st.A2, st.a1 * (1 + sliderDa.value / 100), st.a2
    paramsNom = st.g, st.A1, st.A2, st.a1, st.a2

    x0 = [y0 + sliderDh.value, stationarySys(y0, paramsNom)[1]]

    res = solve_ivp(closedLoop,
                    [tSim[0], tSim[-1]],
                    x0,
                    t_eval=tSim,
                    max_step=1,
                    args=(mode, k, ffArgs, params))

    yRef = np.array([trajectory(t, *ffArgs)[0] for t in res.t])
    uRef = np.array([feedForward(t, *ffArgs)[1] for t in res.t])
    uAll = np.maximum([control(t, res.y[:, idx], mode, k, ffArgs) for idx, t in enumerate(res.t)], 0)
    etaS = stationarySys(yRef, paramsNom)[1]

    h1Ref.set_data(res.t, yRef)
    h1Sys.set_data(res.t, res.y[0])
    h2Stat.set_data(res.t, etaS)
    h2Sys.set_data(res.t, res.y[1])
    uFF.set_data(res.t, 1e3 * uRef)
    uSys.set_data(res.t, 1e3 * uAll)

    def limits(ax, *data):
        lo = np.min([np.min(d) for d in data])
        hi = np.max([np.max(d) for d in data])
        ax.set_ylim(lo - 0.1 * (hi - lo) - 1e-3, hi + 0.1 * (hi - lo) + 1e-3)

    limits(ax1, yRef, res.y[0])
    limits(ax2, etaS, res.y[1])
    limits(ax3, 1e3 * uRef, 1e3 * uAll)

    fig.canvas.draw()

sliderYd.observe(calcSystem, names='value')
sliderT.observe(calcSystem, names='value')
sliderK.observe(calcSystem, names='value')
sliderDh.observe(calcSystem, names='value')
sliderDa.observe(calcSystem, names='value')
radioU.observe(calcSystem, names='value')

calcSystem(None)

bFeedForward = VBox([Label(value='Steuerung'), sliderYd, sliderT])
bFeedForward.layout = make_box_layout()
bFeedBack = VBox([Label(value='Regler'), radioU, sliderK])
bFeedBack.layout = make_box_layout()
bModel = VBox([Label(value='Modell'), sliderDh, sliderDa])
bModel.layout = make_box_layout()

controls = HBox([bFeedForward, bFeedBack, bModel])
VBox([controls, output], layout=Layout(display='flex', flex_flow='column', justify_content='center', align_items='center'))